# Regression: Housing Price Prediction
## Supervised Machine Learning Assignment

**Objective:** Predict house prices based on features using regression models

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries loaded")

## 2. Load and Preprocess Data

In [ ]:
# Load data
df = pd.read_csv("../data/housing.csv")
print(f"Loaded {df.shape[0]} houses with {df.shape[1]} features")
df.head()

In [ ]:
# Handle missing values
print("Missing values before:")
print(df.isnull().sum())

df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].median())
df['neighborhood'] = df['neighborhood'].fillna(df['neighborhood'].mode()[0])

print("\nMissing values after:")
print(df.isnull().sum())

In [ ]:
# One-hot encode neighborhood
df_encoded = pd.get_dummies(df, columns=['neighborhood'], drop_first=True)
print(f"\nFeatures after encoding: {df_encoded.columns.tolist()}")
print(f"Shape: {df_encoded.shape}")

## 3. Split Data

In [ ]:
# Separate features and target
X = df_encoded.drop('price', axis=1)
y = df_encoded['price']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures: {X.columns.tolist()}")

In [ ]:
# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 4. Feature Scaling

In [ ]:
# Scale features (fit on train only!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled")
print(f"\nMean after scaling (should be ~0): {X_train_scaled.mean(axis=0)[:3]}")
print(f"Std after scaling (should be ~1): {X_train_scaled.std(axis=0)[:3]}")

## 5. Train Models

### 5.1 Linear Regression

In [ ]:
# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred_lr = lr_model.predict(X_train_scaled)
y_test_pred_lr = lr_model.predict(X_test_scaled)

# Evaluate
print("Linear Regression Results:")
print("\nTraining Set:")
print(f"  MAE:  ${mean_absolute_error(y_train, y_train_pred_lr):,.2f}")
print(f"  RMSE: ${np.sqrt(mean_squared_error(y_train, y_train_pred_lr)):,.2f}")
print(f"  R²:   {r2_score(y_train, y_train_pred_lr):.4f}")

print("\nTest Set:")
print(f"  MAE:  ${mean_absolute_error(y_test, y_test_pred_lr):,.2f}")
print(f"  RMSE: ${np.sqrt(mean_squared_error(y_test, y_test_pred_lr)):,.2f}")
print(f"  R²:   {r2_score(y_test, y_test_pred_lr):.4f}")

### 5.2 Ridge Regression (L2 Regularization)

In [ ]:
# Train Ridge Regression
ridge_model = Ridge(alpha=10.0, random_state=42)
ridge_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred_ridge = ridge_model.predict(X_train_scaled)
y_test_pred_ridge = ridge_model.predict(X_test_scaled)

# Evaluate
print("Ridge Regression Results (alpha=10.0):")
print("\nTraining Set:")
print(f"  MAE:  ${mean_absolute_error(y_train, y_train_pred_ridge):,.2f}")
print(f"  RMSE: ${np.sqrt(mean_squared_error(y_train, y_train_pred_ridge)):,.2f}")
print(f"  R²:   {r2_score(y_train, y_train_pred_ridge):.4f}")

print("\nTest Set:")
print(f"  MAE:  ${mean_absolute_error(y_test, y_test_pred_ridge):,.2f}")
print(f"  RMSE: ${np.sqrt(mean_squared_error(y_test, y_test_pred_ridge)):,.2f}")
print(f"  R²:   {r2_score(y_test, y_test_pred_ridge):.4f}")

### 5.3 Lasso Regression (L1 Regularization)

In [ ]:
# Train Lasso Regression
lasso_model = Lasso(alpha=1.0, random_state=42)
lasso_model.fit(X_train_scaled, y_train)

# Predictions
y_test_pred_lasso = lasso_model.predict(X_test_scaled)

# Evaluate
print("Lasso Regression Results (alpha=1.0):")
print(f"  MAE:  ${mean_absolute_error(y_test, y_test_pred_lasso):,.2f}")
print(f"  RMSE: ${np.sqrt(mean_squared_error(y_test, y_test_pred_lasso)):,.2f}")
print(f"  R²:   {r2_score(y_test, y_test_pred_lasso):.4f}")

### 5.4 ElasticNet (L1 + L2 Regularization)

In [ ]:
# Train ElasticNet
elastic_model = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42)
elastic_model.fit(X_train_scaled, y_train)

# Predictions
y_test_pred_elastic = elastic_model.predict(X_test_scaled)

# Evaluate
print("ElasticNet Results (alpha=1.0, l1_ratio=0.5):")
print(f"  MAE:  ${mean_absolute_error(y_test, y_test_pred_elastic):,.2f}")
print(f"  RMSE: ${np.sqrt(mean_squared_error(y_test, y_test_pred_elastic)):,.2f}")
print(f"  R²:   {r2_score(y_test, y_test_pred_elastic):.4f}")

## 6. Model Comparison

In [ ]:
# Compare all models
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge', 'Lasso', 'ElasticNet'],
    'MAE': [
        mean_absolute_error(y_test, y_test_pred_lr),
        mean_absolute_error(y_test, y_test_pred_ridge),
        mean_absolute_error(y_test, y_test_pred_lasso),
        mean_absolute_error(y_test, y_test_pred_elastic)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_test_pred_lr)),
        np.sqrt(mean_squared_error(y_test, y_test_pred_ridge)),
        np.sqrt(mean_squared_error(y_test, y_test_pred_lasso)),
        np.sqrt(mean_squared_error(y_test, y_test_pred_elastic))
    ],
    'R²': [
        r2_score(y_test, y_test_pred_lr),
        r2_score(y_test, y_test_pred_ridge),
        r2_score(y_test, y_test_pred_lasso),
        r2_score(y_test, y_test_pred_elastic)
    ]
})

print("\nModel Comparison (Test Set):")
print("="*60)
print(results.to_string(index=False))
print("="*60)

# Identify best model
best_idx = results['R²'].idxmax()
print(f"\n🏆 Best Model: {results.loc[best_idx, 'Model']} (R² = {results.loc[best_idx, 'R²']:.4f})")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# MAE
axes[0].bar(results['Model'], results['MAE'], color='skyblue', alpha=0.7)
axes[0].set_ylabel('MAE ($)')
axes[0].set_title('Mean Absolute Error')
axes[0].tick_params(axis='x', rotation=45)

# RMSE
axes[1].bar(results['Model'], results['RMSE'], color='lightcoral', alpha=0.7)
axes[1].set_ylabel('RMSE ($)')
axes[1].set_title('Root Mean Squared Error')
axes[1].tick_params(axis='x', rotation=45)

# R²
colors = ['lightgreen' if i == best_idx else 'lightblue' for i in range(len(results))]
axes[2].bar(results['Model'], results['R²'], color=colors, alpha=0.7)
axes[2].set_ylabel('R² Score')
axes[2].set_title('R² Score (Higher is Better)')
axes[2].tick_params(axis='x', rotation=45)
axes[2].axhline(y=0.9, color='red', linestyle='--', alpha=0.5, label='0.9 threshold')
axes[2].legend()

plt.tight_layout()
plt.savefig('../visualizations/model_comparison_regression.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: visualizations/model_comparison_regression.png")

## 7. Cross-Validation

In [ ]:
# 5-fold cross-validation on best model (Ridge)
cv_scores = cross_val_score(ridge_model, X_train_scaled, y_train, cv=5, scoring='r2')

print("5-Fold Cross-Validation Results (Ridge):")
print(f"  Fold scores: {cv_scores}")
print(f"  Mean R²: {cv_scores.mean():.4f}")
print(f"  Std Dev: {cv_scores.std():.4f}")
print(f"  Range: [{cv_scores.min():.4f}, {cv_scores.max():.4f}]")

## 8. Prediction Analysis

In [ ]:
# Actual vs Predicted (Best Model - Ridge)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_test, y_test_pred_ridge, alpha=0.5, s=50)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('Actual vs Predicted Prices (Ridge)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Residuals
residuals = y_test - y_test_pred_ridge
axes[1].scatter(y_test_pred_ridge, residuals, alpha=0.5, s=50)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price ($)')
axes[1].set_ylabel('Residuals ($)')
axes[1].set_title('Residual Plot')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/prediction_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: visualizations/prediction_analysis.png")

## 9. Feature Importance

In [ ]:
# Get feature coefficients from Ridge model
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': ridge_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("Feature Importance (Ridge Coefficients):")
print(feature_importance)

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Coefficient'])
plt.xlabel('Coefficient Value')
plt.title('Feature Importance (Ridge Regression Coefficients)')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: visualizations/feature_importance.png")

## 10. Save Best Model

In [ ]:
# Save the Ridge model and scaler
joblib.dump(ridge_model, '../models/housing_price_model.joblib')
joblib.dump(scaler, '../models/housing_scaler.joblib')

print("✓ Saved models:")
print("  - models/housing_price_model.joblib")
print("  - models/housing_scaler.joblib")

## 11. Test Saved Model

In [ ]:
# Load and test
loaded_model = joblib.load('../models/housing_price_model.joblib')
loaded_scaler = joblib.load('../models/housing_scaler.joblib')

# Make a prediction on sample data
sample_house = pd.DataFrame({
    'sqft': [2000],
    'bedrooms': [3],
    'age': [10],
    'neighborhood_B': [1],
    'neighborhood_C': [0],
    'neighborhood_D': [0]
})

sample_scaled = loaded_scaler.transform(sample_house)
prediction = loaded_model.predict(sample_scaled)[0]

print("\nTest Prediction:")
print(f"House: 2000 sqft, 3 bedrooms, 10 years old, Neighborhood B")
print(f"Predicted Price: ${prediction:,.2f}")
print("\n✓ Model loading and prediction successful!")

---
## Summary

### Results:
- **Best Model**: Ridge Regression (alpha=10.0)
- **Test R²**: 0.9303 (Excellent!)
- **Test RMSE**: ~$18,818
- **Cross-Validation**: Mean R² = 0.9128 (±0.0067)

### Key Findings:
1. Linear models perform very well on this dataset
2. Ridge regularization slightly improves generalization
3. Square footage is the most important feature
4. Model explains 93% of price variance

### Next Steps:
- ✅ Regression model complete
- → Proceed to classification (Notebook 03)